In [37]:
from langgraph.graph import StateGraph,START,END
from dotenv import load_dotenv
from typing import TypedDict
from langchain_groq import ChatGroq

In [38]:
load_dotenv()

model = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0.3,
)

In [39]:
class blogstate(TypedDict):
    title :str
    outline :str
    blog : str

In [40]:
graph=StateGraph(blogstate)

In [41]:
def create_outline(state : blogstate) -> blogstate:
    title=state['title']

    prompt=f"generate and outline for a blog on the topic {title}"

    outline=model.invoke(prompt).content

    state['outline']=outline

    return state

In [42]:
def create_blog(state :blogstate) -> blogstate:

    title=state['title']
    outline=state['outline']

    prompt=f"generate a blog for the topic {title} having the outline {outline}"

    blog=model.invoke(prompt).content

    state['blog']=blog

    return state

In [43]:
graph.add_node('outline',create_outline)
graph.add_node('blog',create_blog)

In [44]:
graph.add_edge(START,'outline')
graph.add_edge('outline','blog')
graph.add_edge('blog',END)

In [45]:
workflow=graph.compile()

In [47]:
initial_state={'title':'iron man the best'}
final_state=workflow.invoke(initial_state)
print(final_state)
print(final_state['outline'])
print(final_state['blog'])

{'title': 'iron man the best', 'outline': 'Here\'s a suggested outline for a blog on "Iron Man: The Best":\n\n**I. Introduction**\n\n* Briefly introduce the topic of the blog: Iron Man as the best superhero\n* Mention the reasons why Iron Man stands out among other superheroes\n* Thesis statement: Iron Man is the best superhero due to his intelligence, technological advancements, and relatable personality.\n\n**II. Intelligence and Strategic Thinking**\n\n* Discuss Tony Stark\'s genius-level intellect and how it contributes to his success as Iron Man\n* Provide examples of his strategic thinking, such as:\n\t+ Creating advanced suits of armor\n\t+ Outsmarting villains like Ultron and Thanos\n\t+ Developing innovative solutions to complex problems\n* Explain how his intelligence makes him a valuable asset to the Avengers\n\n**III. Technological Advancements**\n\n* Describe the various technologies developed by Tony Stark, including:\n\t+ The Iron Man suit\n\t+ Repulsor technology\n\t+ A